# 08 - Statistical Validation

This notebook consumes the downstream outputs from Pipe 3 and produces the statistical summaries of the workflow, including bootstrap-based uncertainty summaries, paired deltas, and temporal comparison artifacts.

## Methodological role

This stage supports the robustness layer of the study rather than the primary controlled retriever benchmark.

In [ ]:
# ============================================================
# Instalacao de dependencias para Colab
# ============================================================
!pip install -U -q pip setuptools wheel
!pip uninstall -y -q numpy scipy scikit-learn sentence-transformers transformers faiss-cpu numba umap-learn hdbscan || true
!pip install -U -q numpy==2.0.2 scipy==1.14.1 pandas==2.2.2 scikit-learn==1.6.1 openpyxl pyarrow jedi==0.19.2

import numpy, scipy, pandas, sklearn
print("numpy            :", numpy.__version__)
print("scipy            :", scipy.__version__)
print("pandas           :", pandas.__version__)
print("scikit-learn     :", sklearn.__version__)

print("Dependencias instaladas.")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
import math
import os
import random
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_colwidth", 260)


In [ ]:
# ============================================================
# Configuracao geral
# ============================================================

DRIVE_ROOT = Path("/content/drive/MyDrive/Unicamp")
PROJECT_ROOT = DRIVE_ROOT / "artigo bibliometria" / "grounded-scientometrics-solarphysics-retrieval"
DATA_ROOT = DRIVE_ROOT / "artigo bibliometria" / "base de dados" / "Artigo_Bibliometria Base Bruta" / "BASES_UNIFICADAS_POR_TEMA"

TARGET_CORPUS = "ML_Multimodal"
READ_STAGE = "07_pipe3_agent_scientometrics"
WRITE_STAGE = "08_pipe4_statistical_validation"
PIPE3_ROOT = DATA_ROOT / TARGET_CORPUS / "04_rebuild_outputs" / READ_STAGE
WRITE_ROOT = DATA_ROOT / TARGET_CORPUS / "04_rebuild_outputs" / WRITE_STAGE
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
BOOTSTRAP_ROUNDS = 2000
MANUAL_AUDIT_N = 24

assert PIPE3_ROOT.exists(), f"PIPE3_ROOT nao encontrado: {PIPE3_ROOT}"
print("PIPE3_ROOT =", PIPE3_ROOT)
print("WRITE_ROOT =", WRITE_ROOT)


In [ ]:
# ============================================================
# Saidas, logs e helpers
# ============================================================

PIPE_START_TS = time.time()


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


ensure_dir(WRITE_ROOT)
ensure_dir(WRITE_ROOT / "tables")
ensure_dir(WRITE_ROOT / "reports")
GLOBAL_LOG_DIR = ensure_dir(PROJECT_ROOT / "outputs" / "camada4_logs")
GLOBAL_LOG_FILE = GLOBAL_LOG_DIR / f"08_pipe4_{RUN_TS}.txt"


def elapsed_seconds() -> float:
    return time.time() - PIPE_START_TS


def fmt_seconds(seconds: float) -> str:
    seconds = int(round(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"


def log(message: str) -> None:
    now = datetime.now().strftime("%H:%M:%S")
    prefix = f"[{now} | +{fmt_seconds(elapsed_seconds())}]"
    line = f"{prefix} {message}"
    print(line, flush=True)
    GLOBAL_LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(GLOBAL_LOG_FILE, "a", encoding="utf-8") as fh:
        fh.write(line + "\n")


def stage_banner(title: str) -> None:
    bar = "=" * 96
    log(bar)
    log(title)
    log(bar)


In [ ]:
# ============================================================
# Leitura dos outputs do Pipe 3
# ============================================================

stage_banner("LEITURA DO PIPE 3")

metrics_df = pd.read_csv(PIPE3_ROOT / "tables" / "retriever_query_metrics.csv")
agent_df = pd.read_csv(PIPE3_ROOT / "tables" / "agent_outputs.csv")
hits_df = pd.read_csv(PIPE3_ROOT / "tables" / "retrieval_hits_all.csv")

log(f"[load] metrics={len(metrics_df)} | agent_outputs={len(agent_df)} | hits={len(hits_df)}")
display(metrics_df.head(10))
display(agent_df.head(10))


In [ ]:
# ============================================================
# Estatisticas e intervalos de confianca
# ============================================================

stage_banner("ESTATISTICAS")


def wilson_interval(successes: int, total: int, z: float = 1.96):
    if total == 0:
        return (np.nan, np.nan, np.nan)
    p = successes / total
    denom = 1 + z**2 / total
    centre = (p + z**2 / (2 * total)) / denom
    margin = (z / denom) * math.sqrt((p * (1 - p) / total) + (z**2 / (4 * total**2)))
    return (p, max(0.0, centre - margin), min(1.0, centre + margin))


def bootstrap_mean_ci(values: np.ndarray, rounds: int = BOOTSTRAP_ROUNDS, seed: int = 42):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(rounds, len(values)), replace=True)
    means = samples.mean(axis=1)
    return (
        float(values.mean()),
        float(np.quantile(means, 0.025)),
        float(np.quantile(means, 0.975)),
    )


summary_rows = []
for (period, retriever), sub in metrics_df.groupby(["period", "retriever"], sort=False):
    mean_top1, top1_lo, top1_hi = bootstrap_mean_ci(sub["top1_score"].to_numpy())
    mean_cit, cit_lo, cit_hi = bootstrap_mean_ci(sub["mean_topk_citations"].to_numpy())
    mean_recent, recent_lo, recent_hi = bootstrap_mean_ci(sub["recent_share_topk"].to_numpy())
    success_rate, succ_lo, succ_hi = wilson_interval(
        int(sub["distinct_sources_topk"].fillna(0).ge(3).sum()),
        int(len(sub)),
    )
    summary_rows.append(
        {
            "period": period,
            "retriever": retriever,
            "n_queries": int(len(sub)),
            "mean_top1_score": round(mean_top1, 4),
            "mean_top1_score_ci_low": round(top1_lo, 4),
            "mean_top1_score_ci_high": round(top1_hi, 4),
            "mean_topk_citations": round(mean_cit, 4),
            "mean_topk_citations_ci_low": round(cit_lo, 4),
            "mean_topk_citations_ci_high": round(cit_hi, 4),
            "mean_recent_share": round(mean_recent, 4),
            "mean_recent_share_ci_low": round(recent_lo, 4),
            "mean_recent_share_ci_high": round(recent_hi, 4),
            "share_queries_ge_3_sources": round(success_rate, 4),
            "share_queries_ge_3_sources_ci_low": round(succ_lo, 4),
            "share_queries_ge_3_sources_ci_high": round(succ_hi, 4),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(WRITE_ROOT / "tables" / "statistical_summary.csv", index=False)
display(summary_df)


In [ ]:
# ============================================================
# Deltas principais: especializado vs generico e core vs holdout
# ============================================================

stage_banner("DELTAS PRINCIPAIS")

pivot_top1 = metrics_df.pivot_table(
    index=["period", "query_id", "query_text", "source_corpus"],
    columns="retriever",
    values="top1_score",
    aggfunc="first",
).reset_index()

if "specialized" in pivot_top1.columns and "generic" in pivot_top1.columns:
    pivot_top1["delta_specialized_vs_generic"] = pivot_top1["specialized"] - pivot_top1["generic"]
else:
    pivot_top1["delta_specialized_vs_generic"] = np.nan

delta_rows = []
for period, sub in pivot_top1.groupby("period", sort=False):
    delta_mean, delta_lo, delta_hi = bootstrap_mean_ci(sub["delta_specialized_vs_generic"].to_numpy())
    delta_rows.append(
        {
            "period": period,
            "metric": "delta_specialized_vs_generic_top1",
            "mean": round(delta_mean, 4),
            "ci_low": round(delta_lo, 4),
            "ci_high": round(delta_hi, 4),
            "n_queries": int(len(sub)),
        }
    )

core_holdout_merge = summary_df.pivot(index="retriever", columns="period", values="mean_top1_score").reset_index()
if {"core", "holdout"}.issubset(core_holdout_merge.columns):
    core_holdout_merge["holdout_minus_core_mean_top1"] = core_holdout_merge["holdout"] - core_holdout_merge["core"]

delta_df = pd.DataFrame(delta_rows)
delta_df.to_csv(WRITE_ROOT / "tables" / "paired_deltas.csv", index=False)
core_holdout_merge.to_csv(WRITE_ROOT / "tables" / "core_holdout_mean_top1_delta.csv", index=False)

display(delta_df)
display(core_holdout_merge)


In [ ]:
# ============================================================
# Amostra para auditoria manual pequena
# ============================================================

stage_banner("AUDITORIA MANUAL PEQUENA")

top_hits = hits_df[hits_df["rank"] == 1].copy()
top_hits["audit_bucket"] = top_hits["period"] + "__" + top_hits["retriever"]

audit_rows = []
for bucket, sub in top_hits.groupby("audit_bucket", sort=False):
    take_n = min(max(1, MANUAL_AUDIT_N // max(1, top_hits["audit_bucket"].nunique())), len(sub))
    audit_rows.append(sub.sample(take_n, random_state=42))

audit_df = pd.concat(audit_rows, ignore_index=True) if audit_rows else pd.DataFrame()
audit_df.to_csv(WRITE_ROOT / "tables" / "manual_audit_sample.csv", index=False)

with pd.ExcelWriter(WRITE_ROOT / "reports" / "paper_tables_final.xlsx", engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="statistical_summary", index=False)
    delta_df.to_excel(writer, sheet_name="paired_deltas", index=False)
    core_holdout_merge.to_excel(writer, sheet_name="core_holdout_top1", index=False)
    audit_df.to_excel(writer, sheet_name="manual_audit_sample", index=False)

display(audit_df.head(20))


In [ ]:
# ============================================================
# Manifesto final da validacao
# ============================================================

stage_banner("MANIFESTO FINAL")

manifest_rows = []
for path in sorted(WRITE_ROOT.rglob("*")):
    if path.is_file():
        manifest_rows.append({"artifact": str(path), "size_bytes": path.stat().st_size})

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(WRITE_ROOT / "reports" / "artifact_manifest.csv", index=False)

with open(WRITE_ROOT / "reports" / "validation_manifest.json", "w", encoding="utf-8") as fh:
    json.dump(
        {
            "read_stage": READ_STAGE,
            "write_stage": WRITE_STAGE,
            "bootstrap_rounds": BOOTSTRAP_ROUNDS,
            "manual_audit_n": MANUAL_AUDIT_N,
            "run_ts": RUN_TS,
        },
        fh,
        indent=2,
        ensure_ascii=False,
    )

display(manifest_df.head(30))
print("Arquivos finais salvos em:", WRITE_ROOT)
